# Classical model

In [9]:
import numpy as np
import random
import time
import json
from datetime import datetime
import pennylane as qp
import networkx as nx
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from scipy.linalg import expm

In [10]:
k = L = 2              # degree of the modeled polynomial function
DATASET_SIZE = 500                 # size of the full dataset
num_nodes = 5

TEST_SIZE=0.2
SEED = 10101010
random.seed(SEED)
np.random.seed(SEED)

In [3]:
config = {
    "experiment_name": "classical_baseline",
    "num_nodes": num_nodes,
    "num_layers": L,
    "dataset_size": DATASET_SIZE,
    "test_size": TEST_SIZE,
    "seed": SEED
}

In [11]:
def get_encoding_matrix(adj_matrix):
    """
    Returns the Ising Hamiltonian for a given adjacency matrix, as defined in the project instructions.
    If the graph has no edges, returns the zero Hamiltonian.
    """
    n = adj_matrix.shape[0]
    coeffs = []
    obs = []
    for i in range(n):
        for j in range(i + 1, n):
            if adj_matrix[i, j] == 1:
                coeffs.append(1.)
                obs.append(qp.PauliZ(i) @ qp.PauliZ(j))
    if len(obs) == 0:
        return np.eye(2**adj_matrix.shape[0], dtype=complex)
    H_dense = qp.matrix(qp.Hamiltonian(coeffs, obs), wire_order=range(adj_matrix.shape[0]))
    return expm(1j * H_dense)

### Data Generation

In [12]:
dataset = []

threshold = np.log(num_nodes) / num_nodes   # The theoretical threshold for a graph to be connected is ln(n)/n
edge_probabilities = [random.uniform(threshold*0.9, threshold*1.5) for _ in range(DATASET_SIZE)]

connected_count = 0
disconnected_count = 0
for p in edge_probabilities:
    G = nx.erdos_renyi_graph(n=num_nodes, p=p)
    A = nx.to_numpy_array(G)
    H_G = get_encoding_matrix(A)
    if nx.is_connected(G):
        y = 0
        connected_count+=1
    else:
        y = 1
        disconnected_count+=1
    dataset.append((A, H_G, y))
config["connected_ratio"] = connected_count / len(dataset)

print(f"The full dataset contains {connected_count} connected graphs and {disconnected_count} disconnected graphs.")
train_data, test_data = train_test_split(dataset, test_size=TEST_SIZE, random_state=SEED)

The full dataset contains 229 connected graphs and 271 disconnected graphs.


In [13]:
def filter_isomorphisms(source_dataset):
    """Filters a dataset to contain only non-isomorphic graphs. See the project report for more details and motivation."""

    unique_graphs = list()
    filtered_dataset = []

    for data_point in source_dataset:
        A, _, _ = data_point
        G_current = nx.from_numpy_array(A)

        is_iso = False
        for G_seen in unique_graphs:
            if nx.is_isomorphic(G_current, G_seen):
                is_iso = True
                break

        if not is_iso:
            unique_graphs.append(G_current)
            filtered_dataset.append(data_point)

    return filtered_dataset

print(f"Original train size: {len(train_data)}")
train_data_clean = filter_isomorphisms(train_data)
print(f"Cleaned train size (no internal isomorphisms): {len(train_data_clean)}\n")

train_data = train_data_clean
X_train = np.array([data_point[0].flatten() for data_point in train_data])
X_train = np.array([np.concatenate([data_point[1].flatten().real,data_point[1].flatten().imag])
    for data_point in train_data])
y_train = np.array([data_point[2] for data_point in train_data])
X_test = np.array([data_point[0].flatten() for data_point in test_data])
X_test = np.array([np.concatenate([data_point[1].flatten().real,data_point[1].flatten().imag])
    for data_point in test_data])
y_test = np.array([data_point[2] for data_point in test_data])

Original train size: 400
Cleaned train size (no internal isomorphisms): 31



### Classical polynomial model

In [14]:
# - PolynomialFeatures generates all combinations of features up to degree k
# - LogisticRegression applies the free weights (w) and a sigmoid for binary classification
classical_model = make_pipeline(
    PolynomialFeatures(degree=L, include_bias=False),
    # penalty=None ensures the coefficients w are "completely free"
    LogisticRegression(fit_intercept=False, penalty=None, max_iter=1000)
)

### Training

In [15]:
print("Training the classical model...")

start_time = time.time()

classical_model.fit(X_train, y_train)

end_time = time.time()
duration = end_time - start_time
print(duration)

Training the classical model...
1.1304810047149658


### Performance Evaluation

In [16]:
train_acc = accuracy_score(y_train, classical_model.predict(X_train))
test_acc = accuracy_score(y_test, classical_model.predict(X_test))

print(f"Training accuracy: {train_acc:.4f}")
print(f"Testing accuracy:  {test_acc:.4f}")

Training accuracy: 1.0000
Testing accuracy:  0.7400


In [17]:
log_reg_model = classical_model.named_steps['logisticregression']
num_params = log_reg_model.coef_.size

print(f"The model has {num_params} free parameters.")

The model has 2100224 free parameters.


In [ ]:
def generate_novel_test_set(train_data, num_nodes, num_new_graphs):
    """
    Generates a dataset of non-isomorphic graphs that do not
    share an isomorphism class with any graph in the used training set.
    """
    train_graphs = [nx.from_numpy_array(item[0]) for item in train_data]

    test_dataset = []
    test_graphs = []

    threshold = np.log(num_nodes) / num_nodes

    while len(test_dataset) < num_new_graphs:
        p = random.uniform(threshold * 0.9, threshold * 1.5)
        G_cand = nx.erdos_renyi_graph(n=num_nodes, p=p)

        is_iso_train = any(nx.is_isomorphic(G_cand, G_t) for G_t in train_graphs)
        is_iso_test = any(nx.is_isomorphic(G_cand, G_test) for G_test in test_graphs)

        if not is_iso_train and not is_iso_test:
            A_cand = nx.to_numpy_array(G_cand)
            H_G_cand = get_encoding_matrix(A_cand)
            y_cand = 0 if nx.is_connected(G_cand) else 1

            test_dataset.append((A_cand, H_G_cand, y_cand))
            test_graphs.append(G_cand)

    print(f"Generated {len(test_dataset)} novel graphs.")
    return test_dataset

# Example usage:
novel_test_data = generate_novel_test_set(train_data, num_nodes, num_new_graphs=100)
X_novel = np.array([data_point[0].flatten() for data_point in novel_test_data])
#X_novel = np.array([np.concatenate([data_point[1].flatten().real,data_point[1].flatten().imag])
    #for data_point in novel_test_data])
y_novel = np.array([data_point[2] for data_point in novel_test_data])
novel_test_acc = accuracy_score(y_novel, classical_model.predict(X_novel))
print(novel_test_acc)

In [ ]:
results = {
    **config,
    "num_params": num_params,
    "final_train_acc": round(float(train_acc), 3),
    "final_test_acc": round(float(test_acc), 3),
    "novel_test_acc": round(float(novel_test_acc), 3),
    "total_training_time_sec": round(duration, 2),
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

with open("experiments_log.jsonl", "a") as f:
    f.write(json.dumps(results) + "\n")